# Phase 5 - Prediction cube for the interactive explorer

Exports a compact grid of predictions so a static page can show, for a chosen protein
and chosen conditions, what the model predicts **under each generalization regime**.

Four layers are computed for every protein:

| Layer | Model | What it represents |
|---|---|---|
| **A** | trained on all data (constrained) | the protein is characterised - regime A, ROC-AUC 0.834 |
| **B** | trained with this sequence held out (constrained) | the protein is new - regime B, 0.786 |
| **C\*** | trained with this sequence's homology group held out | the family is new - regime C\*, 0.659 |
| **U** | trained on all data, **unconstrained** | the physics comparison - this one can dip |

Switching between A, B and C\* on the same protein shows the 0.175 ROC-AUC spread as a
change in the predicted phase boundary. Switching to U shows the non-physical pockets.

**Setup:** T4 GPU, ~15 min. Upload `llpsense_final_noseq.csv` and `llpsense_t5.npz`.

## 1. Setup

In [ ]:
import numpy as np, pandas as pd, subprocess, json, time, base64, warnings
warnings.filterwarnings("ignore")
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedGroupKFold
GPU = subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0

from google.colab import files
up = files.upload()
df  = pd.read_csv("llpsense_final_noseq.csv")
z   = np.load("llpsense_t5.npz", allow_pickle=True)
emb = pd.DataFrame(z["X"], index=z["SeqID"])
COND = ['Conc','pH','temp','MgCl2','NaCl','KCl','PEG300-1k','PEG3k-6k','PEG8k-20k',
        'Ficoll','Dextran -40','Dextran 70-','Glycerol']
X = np.hstack([emb.loc[df.SeqID].to_numpy(np.float32), df[COND].to_numpy(np.float32)])
y, w = df.Label.to_numpy(), df.w_record.to_numpy()
N_EMB = emb.shape[1]; IDX = {c: N_EMB+i for i,c in enumerate(COND)}
UP = ['Conc','PEG300-1k','PEG3k-6k','PEG8k-20k','Ficoll','Dextran -40','Dextran 70-']
MONO = tuple([0]*N_EMB + [1 if c in UP else 0 for c in COND])
HP = dict(n_estimators=877, max_depth=14, learning_rate=0.023722174554030036,
          min_child_weight=2, colsample_bytree=0.5655509178160505,
          subsample=0.5482175662874387, gamma=0.0012682360748828933,
          reg_lambda=0.3024802937837069, reg_alpha=0.04238171400820899, random_state=42)
RT = dict(tree_method="hist", eval_metric="logloss",
          **({"device":"cuda"} if GPU else {"n_jobs":-1}))
print("ready", X.shape)

## 2. Choose the proteins

Proteins with the most experimental records, so the explorer is populated with
well-characterised systems rather than one-off constructs.

In [ ]:
N_PROT = 36
cnt = (df.groupby("SeqID")
         .agg(name=("ProteinName","first"), records=("RecordID","nunique"),
              pos=("Label","mean"))
         .sort_values("records", ascending=False))
sel = cnt.head(N_PROT).copy()
# representative condition row index for each chosen sequence
rep_idx = {s: df.index[df.SeqID == s][0] for s in sel.index}
print(sel.head(12).to_string())
print(f"\n{len(sel)} proteins selected, {sel.records.sum()} records covered")

## 3. Grid definition

Concentration is log-spaced over the data support; salt linear to 1000 mM. Temperature
and crowder are small discrete sets, giving six condition panels per protein.

In [ ]:
CONC_uM = np.logspace(np.log10(0.5), np.log10(563.0), 32)
NACL_mM = np.linspace(0.0, 1000.0, 16)
TEMP_C  = [4.0, 25.0, 37.0]
PEG_pct = [0.0, 8.0]
MAXV = dict(Conc=1000., NaCl=2000., temp=60., PEG=50.)
SHAPE = (len(TEMP_C), len(PEG_pct), len(NACL_mM), len(CONC_uM))
print("grid per protein per layer:", SHAPE, "=", int(np.prod(SHAPE)), "points")

def grid_rows(base_row):
    """expand one condition row into the full grid, in (temp, peg, nacl, conc) order"""
    n = int(np.prod(SHAPE))
    G = np.repeat(base_row[None, :], n, axis=0)
    k = 0
    for t in TEMP_C:
        for p in PEG_pct:
            for s in NACL_mM:
                for c in CONC_uM:
                    G[k, IDX['temp']]      = t / MAXV['temp']
                    G[k, IDX['PEG3k-6k']]  = p / MAXV['PEG']
                    G[k, IDX['NaCl']]      = s / MAXV['NaCl']
                    G[k, IDX['Conc']]      = c / MAXV['Conc']
                    k += 1
    return G

GRIDS = {s: grid_rows(X[rep_idx[s]].copy()) for s in sel.index}
print("grids built for", len(GRIDS), "proteins")

## 4. Train the four layers

In [ ]:
cube = {s: {} for s in sel.index}
t0 = time.time()

# --- layers A and U: trained on everything ---
for tag, kw in [("A", {"monotone_constraints": MONO}), ("U", {})]:
    m = XGBClassifier(**HP, **kw, **RT).fit(X, y, sample_weight=w)
    for s in sel.index:
        cube[s][tag] = m.predict_proba(GRIDS[s])[:, 1]
    print(f"layer {tag} done  ({time.time()-t0:.0f}s)", flush=True)

# --- layers B and C*: the protein is held out ---
for tag, key in [("B", "SeqID"), ("C", "GroupX")]:
    g = df[key].to_numpy()
    done = set()
    for i, j in StratifiedGroupKFold(5, shuffle=True, random_state=1).split(X, y, groups=g):
        held = set(df[key].to_numpy()[j])
        targets = [s for s in sel.index
                   if s not in done and df.loc[rep_idx[s], key] in held]
        if not targets:
            continue
        m = XGBClassifier(**HP, monotone_constraints=MONO, **RT).fit(X[i], y[i], sample_weight=w[i])
        for s in targets:
            cube[s][tag] = m.predict_proba(GRIDS[s])[:, 1]
            done.add(s)
    missing = [s for s in sel.index if s not in done]
    print(f"layer {tag} done, {len(done)}/{len(sel)} covered"
          f"{', missing ' + str(len(missing)) if missing else ''}  ({time.time()-t0:.0f}s)", flush=True)
    for s in missing:                       # fallback so the page never has holes
        cube[s][tag] = cube[s]["A"]

## 5. Export

Probabilities are quantised to one byte each (0-255) and base64-encoded, which keeps
the file small enough to embed directly in a self-contained page.

In [ ]:
payload = {
  "axes": {"conc_uM": [round(float(v),3) for v in CONC_uM],
           "nacl_mM": [round(float(v),1) for v in NACL_mM],
           "temp_C": TEMP_C, "peg_pct": PEG_pct},
  "shape": list(SHAPE),
  "layers": {
     "A": {"label": "Protein characterised (regime A)", "auc": 0.834},
     "B": {"label": "New protein (regime B)",            "auc": 0.786},
     "C": {"label": "New protein family (regime C*)",    "auc": 0.659},
     "U": {"label": "Unconstrained model (physics comparison)", "auc": 0.834}},
  "proteins": []
}
for s in sel.index:
    row = sel.loc[s]
    obs = df[(df.SeqID == s) & (df.Label == 1)].Conc * 1000.0
    entry = {"id": s, "name": str(row["name"]), "records": int(row["records"]),
             "pos_rate": round(float(row["pos"]), 3),
             "min_obs_pos_uM": (round(float(obs.min()), 3) if len(obs) else None),
             "data": {}}
    for tag in ["A", "B", "C", "U"]:
        q = np.clip(np.round(cube[s][tag] * 255), 0, 255).astype(np.uint8)
        entry["data"][tag] = base64.b64encode(q.tobytes()).decode("ascii")
    payload["proteins"].append(entry)

with open("explorer_data.json", "w") as f:
    json.dump(payload, f, separators=(",", ":"))
import os
print("explorer_data.json  %.2f MB" % (os.path.getsize("explorer_data.json")/1e6))
print("proteins:", len(payload["proteins"]), "| points per protein per layer:", int(np.prod(SHAPE)))

from google.colab import files
files.download("explorer_data.json")

### Then

Send `explorer_data.json` back and the interactive page gets built around it: pick a
protein, drag salt / temperature / crowder, and watch the predicted phase boundary move
- with a regime switch that shows how much of the prediction survives when the protein,
or its whole family, is unknown to the model.